# IntZ Example 14: Three-Epoch Kinematic Arc

**EPS Research IntZ Corpus v1.0** | Flynn, D.C. (2026)

Compares observed circular velocity (Vc), velocity dispersion (σ₀), and
kinematic ratio V/σ across three cosmic epochs using the EPS Research corpora:

- **z = 0:** Unified HI Corpus v7.0 (SPARC, 175 galaxies)
- **z ~ 0.6–1.0:** IntZ Corpus v1.0, KROSS Tier-1 (166 galaxies)
- **z ~ 4–6:** High-z Kinematic Corpus Z1 (8 tier-1 rotators)

This notebook uses **observed kinematics only** — no omega correction applied.


In [ ]:
import csv
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# ── Load IntZ KROSS Tier-1 ────────────────────────────────────────────────
intz_rows = []
with open('intz_corpus_v1b_flat.csv') as f:
    for row in csv.DictReader(f):
        if row['survey'] == 'KROSS' and row['quality_tier'] == '1':
            try:
                intz_rows.append({
                    'z':     float(row['z_spec']),
                    'Vc':    float(row['Vc_kms']),
                    'sigma': float(row['sigma0_kms']),
                    'vos':   float(row['v_over_sigma']),
                })
            except (ValueError, KeyError):
                pass

print(f'KROSS Tier-1: {len(intz_rows)} galaxies')

# ── Load Z1 tier-1 rotators ───────────────────────────────────────────────
with open('../../highz/high_z_kinematic_corpus_Z1.json') as f:
    z1 = json.load(f)
z1_rows = []
for g in z1['galaxies']:
    if g.get('is_rotator') and g.get('quality_tier') == 1:
        z1_rows.append({
            'z':   g['redshift'],
            'Vc':  g.get('vrot_max_kms') or g.get('vrot_mean_kms'),
            'sigma': g.get('sigma_mean_kms'),
            'vos': g.get('v_over_sigma'),
        })
print(f'Z1 Tier-1:    {len(z1_rows)} galaxies')

# ── SPARC z=0 reference (Flynn & Cannaliato 2025) ─────────────────────────
sparc_vc_mean,  sparc_vc_std  = 120.5, 45.2   # representative SPARC values
sparc_vos_mean, sparc_vos_std =   8.2,  3.1   # V/sigma at z=0 (rotation dominated)


In [ ]:
# ── Figure: Three kinematic quantities vs redshift ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

kross_z   = [r['z']   for r in intz_rows]
kross_vc  = [r['Vc']  for r in intz_rows if r['Vc']]
kross_sig = [r['sigma'] for r in intz_rows if r['sigma']]
kross_vos = [r['vos'] for r in intz_rows if r['vos'] and float(r['vos']) < 20]

z1_z   = [r['z']   for r in z1_rows]
z1_vc  = [r['Vc']  for r in z1_rows if r['Vc']]
z1_vos = [r['vos'] for r in z1_rows if r['vos']]

# Panel 1: Vc vs z
ax = axes[0]
ax.axhspan(sparc_vc_mean-sparc_vc_std, sparc_vc_mean+sparc_vc_std,
           alpha=0.2, color='#2ca02c', label='SPARC z=0 (±1σ)')
ax.axhline(sparc_vc_mean, color='#2ca02c', lw=2, ls='--')
ax.scatter(kross_z, kross_vc[:len(kross_z)], s=8, alpha=0.4,
           color='#ff7f0e', label='KROSS z~0.9')
ax.scatter(z1_z,   z1_vc,   s=60, color='#d62728',
           marker='D', edgecolors='k', lw=0.5, label='Z1 z~4-6')
ax.set_xlabel('Redshift z', fontsize=11)
ax.set_ylabel(r'$V_c$ (km/s)', fontsize=11)
ax.set_title('Circular Velocity', fontsize=11)
ax.legend(fontsize=8)

# Panel 2: σ vs z
ax2 = axes[1]
ax2.scatter(kross_z, kross_sig[:len(kross_z)], s=8, alpha=0.4,
            color='#ff7f0e', label='KROSS Tier-1')
ax2.set_xlabel('Redshift z', fontsize=11)
ax2.set_ylabel(r'$\sigma_0$ (km/s)', fontsize=11)
ax2.set_title('Velocity Dispersion', fontsize=11)
ax2.legend(fontsize=8)

# Panel 3: V/σ vs z
ax3 = axes[2]
ax3.axhspan(sparc_vos_mean-sparc_vos_std, sparc_vos_mean+sparc_vos_std,
            alpha=0.2, color='#2ca02c', label='SPARC z=0 (±1σ)')
ax3.axhline(sparc_vos_mean, color='#2ca02c', lw=2, ls='--')
ax3.axhline(1.0, color='black', lw=1, ls=':', alpha=0.5, label='V/σ = 1')
ax3.scatter(kross_z, kross_vos[:len(kross_z)], s=8, alpha=0.4,
            color='#ff7f0e', label='KROSS z~0.9')
ax3.scatter(z1_z, z1_vos, s=60, color='#d62728',
            marker='D', edgecolors='k', lw=0.5, label='Z1 z~4-6')
ax3.set_xlabel('Redshift z', fontsize=11)
ax3.set_ylabel(r'$V/\sigma$', fontsize=11)
ax3.set_title('Kinematic State V/σ', fontsize=11)
ax3.legend(fontsize=8)

plt.suptitle('Three-Epoch Kinematic Arc — EPS Research Corpora\n'
             'Observed kinematics (no omega correction)', fontsize=11)
plt.tight_layout()
plt.savefig('intz_nb14_three_epoch_arc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: intz_nb14_three_epoch_arc.png')
print(f'KROSS median Vc:  {np.median(kross_vc):.1f} km/s')
print(f'KROSS median V/σ: {np.median(kross_vos):.2f}')
